# Decision Tree Pruning and Model Complexity

In this notebook, we'll explore the critical concepts of decision tree pruning and model complexity. We'll implement both pre-pruning and post-pruning techniques from scratch, analyze the bias-variance tradeoff, and demonstrate how proper pruning can significantly improve generalization performance.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error
import seaborn as sns
sns.set_style("whitegrid")

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

## Section 1: Motivation - Why Deep Trees Overfit

Decision trees have the remarkable ability to perfectly memorize training data when allowed to grow deep enough. While this might seem like an advantage, it leads to severe overfitting. Let's first visualize this phenomenon.

In [ ]:
# Generate synthetic dataset for classification
X, y = make_classification(n_samples=300, n_features=2, n_redundant=0, 
                          n_informative=2, n_clusters_per_class=1, 
                          random_state=42)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
def gini_impurity(y):
    """Calculate Gini impurity for a given array of labels."""
    classes, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)
    gini = 1 - np.sum(probabilities ** 2)
    return gini

def entropy(y):
    """Calculate entropy for a given array of labels."""
    classes, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)
    # Add small epsilon to avoid log(0)
    probabilities = np.clip(probabilities, 1e-15, 1.0)
    entropy_val = -np.sum(probabilities * np.log2(probabilities))
    return entropy_val

def information_gain(parent, left_child, right_child, criterion='gini'):
    """Calculate information gain from splitting parent into left and right children."""
    if criterion == 'gini':
        score_func = gini_impurity
    else:
        score_func = entropy
        
    parent_score = score_func(parent)
    
    # Calculate weighted scores for children
    total_samples = len(left_child) + len(right_child)
    left_weight = len(left_child) / total_samples
    right_weight = len(right_child) / total_samples
    
    weighted_score = (left_weight * score_func(left_child) + 
                     right_weight * score_func(right_child))
    
    return parent_score - weighted_score

In [ ]:
class DecisionTreeNode:
    def __init__(self):
        self.feature_idx = None
        self.threshold = None
        self.left = None
        self.right = None
        self.value = None  # For leaf nodes
        self.samples = None
        self.impurity = None
        self.n_classes = None

In [ ]:
class DecisionTreeClassifier:
    def __init__(self, max_depth=None, min_samples_split=2, min_samples_leaf=1, 
                 max_features=None, criterion='gini', random_state=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.criterion = criterion
        self.random_state = random_state
        if random_state is not None:
            np.random.seed(random_state)
        
        self.root = None
        self.n_classes = None
        self.n_features = None
        
    def _best_split(self, X, y):
        """Find the best split for the current node."""
        best_gain = -1
        best_feature_idx = None
        best_threshold = None
        
        n_features = X.shape[1]
        
        # Consider only a random subset of features if max_features is set
        if self.max_features is not None:
            feature_indices = np.random.choice(n_features, 
                                            size=min(self.max_features, n_features), 
                                            replace=False)
        else:
            feature_indices = range(n_features)
        
        for feature_idx in feature_indices:
            thresholds = np.unique(X[:, feature_idx])
            
            for threshold in thresholds:
                left_mask = X[:, feature_idx] <= threshold
                right_mask = ~left_mask
                
                if np.sum(left_mask) < self.min_samples_leaf or np.sum(right_mask) < self.min_samples_leaf:
                    continue
                
                gain = information_gain(y, y[left_mask], y[right_mask], self.criterion)
                
                if gain > best_gain:
                    best_gain = gain
                    best_feature_idx = feature_idx
                    best_threshold = threshold
        
        return best_feature_idx, best_threshold, best_gain
    
    def _build_tree(self, X, y, depth=0):
        """Recursively build the decision tree."""
        n_samples, n_features = X.shape
        n_classes = len(np.unique(y))
        
        # Create a node
        node = DecisionTreeNode()
        node.samples = n_samples
        node.n_classes = n_classes
        node.impurity = gini_impurity(y) if self.criterion == 'gini' else entropy(y)
        
        # Stopping criteria
        if (self.max_depth is not None and depth >= self.max_depth) or \
           n_samples < self.min_samples_split or \
           n_classes == 1:
            # Create leaf node
            classes, counts = np.unique(y, return_counts=True)
            node.value = classes[np.argmax(counts)]
            return node
        
        # Find best split
        feature_idx, threshold, gain = self._best_split(X, y)
        
        if feature_idx is None or gain <= 0:
            # No good split found, make leaf
            classes, counts = np.unique(y, return_counts=True)
            node.value = classes[np.argmax(counts)]
            return node
        
        # Make split
        node.feature_idx = feature_idx
        node.threshold = threshold
        
        left_mask = X[:, feature_idx] <= threshold
        right_mask = ~left_mask
        
        # Recursively build left and right subtrees
        node.left = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        node.right = self._build_tree(X[right_mask], y[right_mask], depth + 1)
        
        return node
    
    def fit(self, X, y):
        """Fit the decision tree to the training data."""
        self.n_classes = len(np.unique(y))
        self.n_features = X.shape[1]
        self.root = self._build_tree(X, y)
        return self
    
    def _predict_sample(self, x, node):
        """Predict a single sample using the trained tree."""
        if node.value is not None:  # Leaf node
            return node.value
        
        if x[node.feature_idx] <= node.threshold:
            return self._predict_sample(x, node.left)
        else:
            return self._predict_sample(x, node.right)
    
    def predict(self, X):
        """Predict class labels for samples in X."""
        predictions = []
        for x in X:
            pred = self._predict_sample(x, self.root)
            predictions.append(pred)
        return np.array(predictions)
    
    def get_depth(self, node=None):
        """Get the depth of the tree."""
        if node is None:
            node = self.root
        if node.value is not None:  # Leaf node
            return 0
        return 1 + max(self.get_depth(node.left), self.get_depth(node.right))
    
    def count_nodes(self, node=None):
        """Count the number of nodes in the tree."""
        if node is None:
            node = self.root
        if node.value is not None:  # Leaf node
            return 1
        return 1 + self.count_nodes(node.left) + self.count_nodes(node.right)

In [ ]:
# Demonstrate overfitting with trees of different depths
max_depths = [1, 3, 5, 10, None]
train_accuracies = []
test_accuracies = []

for max_d in max_depths:
    tree = DecisionTreeClassifier(max_depth=max_d, random_state=42)
    tree.fit(X_train, y_train)
    
    train_pred = tree.predict(X_train)
    test_pred = tree.predict(X_test)
    
    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)
    
    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)
    
    print(f"Max Depth: {max_d}, Train Acc: {train_acc:.3f}, Test Acc: {test_acc:.3f}")

In [ ]:
# Visualize decision boundaries for different depths
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for i, max_d in enumerate([1, 3, 5, 10, None]):
    tree = DecisionTreeClassifier(max_depth=max_d, random_state=42)
    tree.fit(X_train, y_train)
    
    # Create a mesh for plotting decision boundary
    h = 0.02
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    Z = tree.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    axes[i].contourf(xx, yy, Z, alpha=0.4, cmap=plt.cm.RdYlBu)
    axes[i].scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap=plt.cm.RdYlBu, edgecolors='black')
    axes[i].set_title(f'Max Depth: {"inf" if max_d is None else max_d}')
    axes[i].set_xlabel('Feature 1')
    axes[i].set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

## Section 2: Tree Depth vs Complexity - The Bias-Variance Tradeoff

The bias-variance tradeoff is a fundamental concept in machine learning. For decision trees, this relationship can be understood mathematically:

$$\text{Expected Error} = \text{Bias}^2 + \text{Variance} + \text{Irreducible Error}$$

Where:
- **Bias** represents the error due to overly simplistic assumptions in the model
- **Variance** represents the error due to sensitivity to small fluctuations in the training set

For decision trees:
- Shallow trees have high bias (underfitting) but low variance
- Deep trees have low bias (can capture complex patterns) but high variance (overfitting)

In [ ]:
# Generate more data to see the bias-variance tradeoff clearly
X_large, y_large = make_classification(n_samples=1000, n_features=2, n_redundant=0, 
                                     n_informative=2, n_clusters_per_class=1, 
                                     random_state=42)
X_train_large, X_test_large, y_train_large, y_test_large = train_test_split(
    X_large, y_large, test_size=0.3, random_state=42)

In [ ]:
# Plot training vs validation curves for different depths
depths = list(range(1, 16))
train_errors = []
val_errors = []

for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train_large, y_train_large)
    
    train_pred = tree.predict(X_train_large)
    val_pred = tree.predict(X_test_large)
    
    train_error = 1 - accuracy_score(y_train_large, train_pred)
    val_error = 1 - accuracy_score(y_test_large, val_pred)
    
    train_errors.append(train_error)
    val_errors.append(val_error)

plt.figure(figsize=(12, 6))
plt.plot(depths, train_errors, label='Training Error', marker='o', linewidth=2)
plt.plot(depths, val_errors, label='Validation Error', marker='s', linewidth=2)
plt.xlabel('Tree Depth')
plt.ylabel('Error Rate')
plt.title('Bias-Variance Tradeoff: Training vs Validation Error')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

As we can see from the plot:
- **Training error** decreases monotonically as depth increases (the tree fits the training data better)
- **Validation error** initially decreases (reducing bias) but then increases (increasing variance)
- There's an optimal depth where validation error is minimized

This demonstrates the classic bias-variance tradeoff in action.

## Section 3: Pre-Pruning Techniques

Pre-pruning (also called early stopping) involves setting constraints during tree construction to prevent the tree from growing too complex. Common pre-pruning parameters include:

1. **max_depth**: Maximum depth of the tree
2. **min_samples_split**: Minimum number of samples required to split an internal node
3. **min_samples_leaf**: Minimum number of samples required to be at a leaf node
4. **max_features**: Number of features to consider when looking for the best split

Let's experiment with these parameters:

In [ ]:
# Experiment with different pre-pruning parameters
def evaluate_pruning_params(X_train, X_test, y_train, y_test, 
                           max_depth=None, min_samples_split=2, 
                           min_samples_leaf=1, max_features=None):
    tree = DecisionTreeClassifier(max_depth=max_depth, 
                                 min_samples_split=min_samples_split,
                                 min_samples_leaf=min_samples_leaf,
                                 max_features=max_features,
                                 random_state=42)
    tree.fit(X_train, y_train)
    
    train_pred = tree.predict(X_train)
    test_pred = tree.predict(X_test)
    
    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)
    depth = tree.get_depth()
    n_nodes = tree.count_nodes()
    
    return train_acc, test_acc, depth, n_nodes

In [ ]:
# Compare different pre-pruning settings
print("Pre-pruning Comparison:")
print("Setting" + " "*25 + "Train Acc  Test Acc  Depth  Nodes")
print("-"*60)

# No pruning
train_acc, test_acc, depth, n_nodes = evaluate_pruning_params(X_train, X_test, y_train, y_test)
print(f"No pruning" + " "*22 + f"{train_acc:.3f}" + " "*7 + f"{test_acc:.3f}" + " "*6 + f"{depth}" + " "*6 + f"{n_nodes}")

# Max depth = 3
train_acc, test_acc, depth, n_nodes = evaluate_pruning_params(X_train, X_test, y_train, y_test, max_depth=3)
print(f"Max depth = 3" + " "*19 + f"{train_acc:.3f}" + " "*7 + f"{test_acc:.3f}" + " "*6 + f"{depth}" + " "*6 + f"{n_nodes}")

# Min samples split = 10
train_acc, test_acc, depth, n_nodes = evaluate_pruning_params(X_train, X_test, y_train, y_test, min_samples_split=10)
print(f"Min samples split = 10" + " "*10 + f"{train_acc:.3f}" + " "*7 + f"{test_acc:.3f}" + " "*6 + f"{depth}" + " "*6 + f"{n_nodes}")

# Min samples leaf = 5
train_acc, test_acc, depth, n_nodes = evaluate_pruning_params(X_train, X_test, y_train, y_test, min_samples_leaf=5)
print(f"Min samples leaf = 5" + " "*12 + f"{train_acc:.3f}" + " "*7 + f"{test_acc:.3f}" + " "*6 + f"{depth}" + " "*6 + f"{n_nodes}")

# Max features = 1
train_acc, test_acc, depth, n_nodes = evaluate_pruning_params(X_train, X_test, y_train, y_test, max_features=1)
print(f"Max features = 1" + " "*15 + f"{train_acc:.3f}" + " "*7 + f"{test_acc:.3f}" + " "*6 + f"{depth}" + " "*6 + f"{n_nodes}")

# Combination of parameters
train_acc, test_acc, depth, n_nodes = evaluate_pruning_params(X_train, X_test, y_train, y_test, 
                                                            max_depth=4, min_samples_split=5, min_samples_leaf=3)
print(f"Combined params" + " "*16 + f"{train_acc:.3f}" + " "*7 + f"{test_acc:.3f}" + " "*6 + f"{depth}" + " "*6 + f"{n_nodes}")

## Section 4: Post-Pruning (Cost Complexity Pruning)

Post-pruning involves first growing a large tree and then removing branches that don't contribute significantly to model performance. Cost complexity pruning uses the following objective function:

$$R_\alpha(T) = R(T) + \alpha |T|$$

Where:
- $R(T)$ is the training error of tree $T$
- $|T|$ is the number of terminal nodes (leaves) in the tree
- $\alpha$ is the complexity parameter that controls the trade-off between training error and tree complexity

The algorithm works by:
1. Growing the largest possible tree
2. Calculating the cost complexity for each possible subtree
3. Selecting the subtree that minimizes $R_\alpha(T)$

In [ ]:
class PrunedDecisionTreeClassifier:
    def __init__(self, max_depth=None, min_samples_split=2, min_samples_leaf=1, 
                 max_features=None, criterion='gini', random_state=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.criterion = criterion
        self.random_state = random_state
        if random_state is not None:
            np.random.seed(random_state)
        
        self.root = None
        self.n_classes = None
        self.n_features = None
        
    def _best_split(self, X, y):
        """Find the best split for the current node."""
        best_gain = -1
        best_feature_idx = None
        best_threshold = None
        
        n_features = X.shape[1]
        
        # Consider only a random subset of features if max_features is set
        if self.max_features is not None:
            feature_indices = np.random.choice(n_features, 
                                            size=min(self.max_features, n_features), 
                                            replace=False)
        else:
            feature_indices = range(n_features)
        
        for feature_idx in feature_indices:
            thresholds = np.unique(X[:, feature_idx])
            
            for threshold in thresholds:
                left_mask = X[:, feature_idx] <= threshold
                right_mask = ~left_mask
                
                if np.sum(left_mask) < self.min_samples_leaf or np.sum(right_mask) < self.min_samples_leaf:
                    continue
                
                gain = information_gain(y, y[left_mask], y[right_mask], self.criterion)
                
                if gain > best_gain:
                    best_gain = gain
                    best_feature_idx = feature_idx
                    best_threshold = threshold
        
        return best_feature_idx, best_threshold, best_gain
    
    def _build_tree(self, X, y, depth=0):
        """Recursively build the decision tree."""
        n_samples, n_features = X.shape
        n_classes = len(np.unique(y))
        
        # Create a node
        node = DecisionTreeNode()
        node.samples = n_samples
        node.n_classes = n_classes
        node.impurity = gini_impurity(y) if self.criterion == 'gini' else entropy(y)
        
        # Stopping criteria
        if (self.max_depth is not None and depth >= self.max_depth) or \
           n_samples < self.min_samples_split or \
           n_classes == 1:
            # Create leaf node
            classes, counts = np.unique(y, return_counts=True)
            node.value = classes[np.argmax(counts)]
            return node
        
        # Find best split
        feature_idx, threshold, gain = self._best_split(X, y)
        
        if feature_idx is None or gain <= 0:
            # No good split found, make leaf
            classes, counts = np.unique(y, return_counts=True)
            node.value = classes[np.argmax(counts)]
            return node
        
        # Make split
        node.feature_idx = feature_idx
        node.threshold = threshold
        
        left_mask = X[:, feature_idx] <= threshold
        right_mask = ~left_mask
        
        # Recursively build left and right subtrees
        node.left = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        node.right = self._build_tree(X[right_mask], y[right_mask], depth + 1)
        
        return node
    
    def fit(self, X, y):
        """Fit the decision tree to the training data."""
        self.n_classes = len(np.unique(y))
        self.n_features = X.shape[1]
        self.root = self._build_tree(X, y)
        return self
    
    def _predict_sample(self, x, node):
        """Predict a single sample using the trained tree."""
        if node.value is not None:  # Leaf node
            return node.value
        
        if x[node.feature_idx] <= node.threshold:
            return self._predict_sample(x, node.left)
        else:
            return self._predict_sample(x, node.right)
    
    def predict(self, X):
        """Predict class labels for samples in X."""
        predictions = []
        for x in X:
            pred = self._predict_sample(x, self.root)
            predictions.append(pred)
        return np.array(predictions)
    
    def get_depth(self, node=None):
        """Get the depth of the tree."""
        if node is None:
            node = self.root
        if node.value is not None:  # Leaf node
            return 0
        return 1 + max(self.get_depth(node.left), self.get_depth(node.right))
    
    def count_nodes(self, node=None):
        """Count the number of nodes in the tree."""
        if node is None:
            node = self.root
        if node.value is not None:  # Leaf node
            return 1
        return 1 + self.count_nodes(node.left) + self.count_nodes(node.right)
    
    def count_leaves(self, node=None):
        """Count the number of leaves in the tree."""
        if node is None:
            node = self.root
        if node.value is not None:  # Leaf node
            return 1
        return self.count_leaves(node.left) + self.count_leaves(node.right)
    
    def calculate_training_error(self, X, y, node=None):
        """Calculate the training error of the tree."""
        if node is None:
            node = self.root
        
        # If it's a leaf node, calculate misclassification rate for samples reaching this node
        if node.value is not None:
            # Count how many samples reach this leaf and how many are misclassified
            pred = node.value
            actual_values, counts = np.unique(y, return_counts=True)
            most_common_actual = actual_values[np.argmax(counts)]
            
            # Misclassified samples
            incorrect = np.sum(y != pred)
            return incorrect
        
        # If it's an internal node, recursively calculate for left and right subtrees
        left_mask = X[:, node.feature_idx] <= node.threshold
        right_mask = ~left_mask
        
        left_error = self.calculate_training_error(X[left_mask], y[left_mask], node.left)
        right_error = self.calculate_training_error(X[right_mask], y[right_mask], node.right)
        
        return left_error + right_error
    
    def cost_complexity_pruning(self, X, y, alpha):
        """Perform cost complexity pruning with given alpha parameter."""
        # First, calculate the error for each node
        self._calculate_node_errors(X, y, self.root)
        
        # Then perform the pruning
        self._prune_subtree(X, y, self.root, alpha)
    
    def _calculate_node_errors(self, X, y, node):
        """Calculate and store the error for each node."""
        if node.value is not None:  # Leaf node
            # Misclassified samples at this leaf
            pred = node.value
            node.error = np.sum(y != pred)
            node.total_samples = len(y)
            return node.error
        
        # Internal node: calculate for subtrees
        left_mask = X[:, node.feature_idx] <= node.threshold
        right_mask = ~left_mask
        
        left_error = self._calculate_node_errors(X[left_mask], y[left_mask], node.left)
        right_error = self._calculate_node_errors(X[right_mask], y[right_mask], node.right)
        
        node.error = left_error + right_error
        node.total_samples = len(y)
        return node.error
    
    def _prune_subtree(self, X, y, node, alpha):
        """Recursively prune subtrees based on cost complexity criterion."""
        if node.value is not None:  # Already a leaf
            return
        
        # Process children first (post-order traversal)
        left_mask = X[:, node.feature_idx] <= node.threshold
        right_mask = ~left_mask
        
        self._prune_subtree(X[left_mask], y[left_mask], node.left, alpha)
        self._prune_subtree(X[right_mask], y[right_mask], node.right, alpha)
        
        # Calculate the error of keeping the subtree vs making it a leaf
        subtree_error = node.error
        
        # Calculate error if we convert this node to a leaf
        classes, counts = np.unique(y, return_counts=True)
        majority_class = classes[np.argmax(counts)]
        leaf_error = np.sum(y != majority_class)
        
        # Calculate the number of leaves in the subtree
        subtree_leaves = self.count_leaves(node)
        
        # Cost complexity of subtree: error + alpha * num_leaves
        subtree_cost = subtree_error + alpha * subtree_leaves
        
        # Cost complexity of leaf: error + alpha * 1
        leaf_cost = leaf_error + alpha * 1
        
        # If converting to leaf is better, do so
        if leaf_cost <= subtree_cost:
            node.value = majority_class
            node.left = None
            node.right = None
            node.feature_idx = None
            node.threshold = None

In [ ]:
# Demonstrate cost complexity pruning
print("Demonstrating Cost Complexity Pruning")
print("="*40)

# First, train a large tree
large_tree = PrunedDecisionTreeClassifier(max_depth=10, random_state=42)
large_tree.fit(X_train, y_train)

print(f"Original tree - Depth: {large_tree.get_depth()}, Leaves: {large_tree.count_leaves()}")

# Evaluate original tree
orig_train_pred = large_tree.predict(X_train)
orig_test_pred = large_tree.predict(X_test)
orig_train_acc = accuracy_score(y_train, orig_train_pred)
orig_test_acc = accuracy_score(y_test, orig_test_pred)
print(f"Original tree - Train Acc: {orig_train_acc:.3f}, Test Acc: {orig_test_acc:.3f}")

# Apply cost complexity pruning with different alpha values
alphas = [0.01, 0.05, 0.1, 0.2, 0.3]

for alpha in alphas:
    pruned_tree = PrunedDecisionTreeClassifier(max_depth=10, random_state=42)
    pruned_tree.fit(X_train, y_train)
    
    # Apply pruning
    pruned_tree.cost_complexity_pruning(X_train, y_train, alpha)
    
    # Evaluate pruned tree
    pruned_train_pred = pruned_tree.predict(X_train)
    pruned_test_pred = pruned_tree.predict(X_test)
    pruned_train_acc = accuracy_score(y_train, pruned_train_pred)
    pruned_test_acc = accuracy_score(y_test, pruned_test_pred)
    
    print(f"Alpha={alpha:4.2f} - Depth: {pruned_tree.get_depth():2d}, Leaves: {pruned_tree.count_leaves():2d}, "
          f"Train Acc: {pruned_train_acc:.3f}, Test Acc: {pruned_test_acc:.3f}")

## Section 5: Bias-Variance Visualization

Let's create a comprehensive visualization showing the relationship between model complexity (tree depth) and the bias-variance components of error.

In [ ]:
# Create multiple datasets to estimate bias and variance
def generate_datasets(n_datasets=10, n_samples=300):
    """Generate multiple datasets from the same distribution."""
    datasets = []
    for i in range(n_datasets):
        X, y = make_classification(n_samples=n_samples, n_features=2, n_redundant=0, 
                                  n_informative=2, n_clusters_per_class=1, 
                                  random_state=i)
        datasets.append((X, y))
    return datasets

# Generate multiple datasets
datasets = generate_datasets(n_datasets=10, n_samples=300)

# Test set to evaluate all models
X_test_bias_var, y_test_bias_var = make_classification(n_samples=1000, n_features=2, n_redundant=0, 
                                                     n_informative=2, n_clusters_per_class=1, 
                                                     random_state=42)

In [ ]:
# Calculate bias and variance for different tree depths
depths = list(range(1, 11))
bias_squared_list = []
variance_list = []
total_error_list = []

for depth in depths:
    predictions = []
    
    # Train model on each dataset and collect predictions on test set
    for X_train_ds, y_train_ds in datasets:
        tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
        tree.fit(X_train_ds, y_train_ds)
        pred = tree.predict(X_test_bias_var)
        predictions.append(pred)
    
    # Convert to numpy array for easier manipulation
    predictions = np.array(predictions)  # Shape: (n_datasets, n_test_samples)
    
    # Calculate bias^2 and variance for each test sample
    # First, find the true labels by training on a large dataset
    X_true, y_true = make_classification(n_samples=5000, n_features=2, n_redundant=0, 
                                        n_informative=2, n_clusters_per_class=1, 
                                        random_state=42)
    true_tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    true_tree.fit(X_true, y_true)
    true_predictions = true_tree.predict(X_test_bias_var)
    
    # Calculate bias^2 and variance
    avg_predictions = np.mean(predictions, axis=0)
    bias_squared = np.mean((avg_predictions != true_predictions).astype(int))
    
    # Variance calculation
    variances = []
    for j in range(len(X_test_bias_var)):
        sample_preds = predictions[:, j]
        # Calculate variance as the probability of disagreement
        unique_vals, counts = np.unique(sample_preds, return_counts=True)
        if len(unique_vals) > 1:
            # Calculate the probability of disagreement
            prob_disagree = 0
            for i in range(len(unique_vals)):
                for k in range(i+1, len(unique_vals)):
                    p_i = counts[i] / len(sample_preds)
                    p_k = counts[k] / len(sample_preds)
                    prob_disagree += p_i * p_k
            variances.append(prob_disagree)
        else:
            variances.append(0)
    
    variance = np.mean(variances)
    total_error = bias_squared + variance
    
    bias_squared_list.append(bias_squared)
    variance_list.append(variance)
    total_error_list.append(total_error)
    
    print(f"Depth {depth}: Bias² = {bias_squared:.3f}, Variance = {variance:.3f}, Total Error = {total_error:.3f}")

In [ ]:
# Plot bias-variance decomposition
plt.figure(figsize=(12, 6))
plt.plot(depths, bias_squared_list, label='Bias²', marker='o', linewidth=2)
plt.plot(depths, variance_list, label='Variance', marker='s', linewidth=2)
plt.plot(depths, total_error_list, label='Total Error', marker='^', linewidth=2)
plt.xlabel('Tree Depth')
plt.ylabel('Error')
plt.title('Bias-Variance Decomposition Across Different Tree Depths')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Section 6: Practical Guidelines for Pruning

### When to Prune

1. **High variance**: When your model performs significantly better on training data than on validation/test data
2. **Large dataset**: With limited data, pruning becomes more important to avoid overfitting
3. **Real-time requirements**: Simpler models are faster for prediction

### Real-world Considerations

1. **Cross-validation**: Use CV to select pruning parameters rather than a single validation set
2. **Domain knowledge**: Sometimes domain expertise can guide the choice of pruning parameters
3. **Cost-sensitive applications**: In some cases, false positives might be more costly than false negatives, affecting how you evaluate pruning effectiveness
4. **Computational efficiency**: Pruned trees are faster both for training and prediction

### Summary

Both pre-pruning and post-pruning are valuable techniques for controlling model complexity:
- **Pre-pruning** is simpler to implement and computationally efficient
- **Post-pruning** allows the tree to discover important structures before removing less useful branches
- The choice often depends on the specific application and computational constraints